In [ ]:
# 1. Clone the repository into the Colab environment
!git clone https://github.com/vaibhav-pratap-singh-iitg/Multimodal-AI-project/

# 2. Navigate into the cloned folder (Use % instead of ! for changing directories in Colab)
%cd Multimodal-AI-project/

# 3. Create and switch to your dedicated branch
!git checkout -b base_model

Cloning into 'Multimodal-AI-project'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 12 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 8.92 KiB | 8.92 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Multimodal-AI-project
Switched to a new branch 'base_model'


In [ ]:
!git config --global user.email "siddhnatpatil2829@gmail.com"
!git config --global user.name "Siddhant"

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset, concatenate_datasets, DatasetDict
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load and pool the Waterbirds dataset
hf_dataset = load_dataset("grodino/waterbirds")

def add_id(example, idx):
    example["image_id"] = idx
    return example

full_dataset = concatenate_datasets([
    hf_dataset["train"],
    hf_dataset["validation"],
    hf_dataset["test"],
])
full_dataset = full_dataset.map(add_id, with_indices=True)

# 40% train / 40% test / 20% val, deterministic
first_split = full_dataset.train_test_split(test_size=0.2, seed=42)
test_dataset = first_split["test"]
train_val_dataset = first_split["train"]

second_split = train_val_dataset.train_test_split(test_size=0.5, seed=42)
train_dataset = second_split["train"]
val_dataset = second_split["test"]

new_hf_dataset = DatasetDict({
    "train": train_dataset,
    "val": val_dataset,
    "test": test_dataset,
})

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class WaterbirdsTorchDataset(Dataset):
    def __init__(self, hf_split, transform):
        self.hf_split = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.hf_split)

    def __getitem__(self, idx):
        item = self.hf_split[idx]
        image = self.transform(item["image"].convert("RGB"))
        label = item["label"]
        image_id = item["image_id"]
        return image, label, image_id

Using device: cuda


Map:   0%|          | 0/11788 [00:00<?, ? examples/s]

In [ ]:
train_data = WaterbirdsTorchDataset(new_hf_dataset["train"], transform)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

model_choice = "resnet50" # change to "vgg16" if needed

if model_choice == "resnet50":
    model = models.resnet50(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, 2)
elif model_choice == "vgg16":
    model = models.vgg16(pretrained=True)
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, 2)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

print(f"Starting fine-tuning with {model_choice}...")
model.train()

for epoch in range(10):
    total_loss = 0
    for batch_idx, (x, y_true, _) in enumerate(train_loader):
        x, y_true = x.to(device), y_true.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y_true)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch+1}], Batch [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}")

# Remember: DO NOT commit this .pt file to GitHub!
torch.save(model.state_dict(), f"waterbirds_{model_choice}.pt")
print(f"Training complete! Weights saved to waterbirds_{model_choice}.pt")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 180MB/s]


Starting fine-tuning with resnet50...
Epoch [1], Batch [0/148], Loss: 0.5276
Epoch [1], Batch [10/148], Loss: 0.2893
Epoch [1], Batch [20/148], Loss: 0.2744
Epoch [1], Batch [30/148], Loss: 0.0944
Epoch [1], Batch [40/148], Loss: 0.2258
Epoch [1], Batch [50/148], Loss: 0.2795
Epoch [1], Batch [60/148], Loss: 0.2235
Epoch [1], Batch [70/148], Loss: 0.1547
Epoch [1], Batch [80/148], Loss: 0.3739
Epoch [1], Batch [90/148], Loss: 0.0767
Epoch [1], Batch [100/148], Loss: 0.2304
Epoch [1], Batch [110/148], Loss: 0.2069
Epoch [1], Batch [120/148], Loss: 0.1675
Epoch [1], Batch [130/148], Loss: 0.1367
Epoch [1], Batch [140/148], Loss: 0.1401
Epoch [2], Batch [0/148], Loss: 0.1083
Epoch [2], Batch [10/148], Loss: 0.0508
Epoch [2], Batch [20/148], Loss: 0.0360
Epoch [2], Batch [30/148], Loss: 0.0268
Epoch [2], Batch [40/148], Loss: 0.0155
Epoch [2], Batch [50/148], Loss: 0.1155
Epoch [2], Batch [60/148], Loss: 0.0267
Epoch [2], Batch [70/148], Loss: 0.0073
Epoch [2], Batch [80/148], Loss: 0.0163

In [ ]:
model.eval() # reuses the same model object already trained above, in memory

all_results = []
splits_to_run = ["val", "test"]

print("Starting inference...")
for split_name in splits_to_run:
    print(f"Processing {split_name} split...")
    subset = WaterbirdsTorchDataset(new_hf_dataset[split_name], transform)
    loader = DataLoader(subset, batch_size=32, shuffle=False)

    for batch_idx, (x, y_true, image_ids) in enumerate(loader):
        x = x.to(device)

        with torch.no_grad():
            outputs = model(x)
            _, preds = torch.max(outputs, 1)

        for i in range(len(y_true)):
            true_lbl = y_true[i].item()
            pred_lbl = preds[i].item()
            img_id = image_ids[i].item()

            all_results.append({
                "image_id": img_id,
                "true_label": true_lbl,
                "pred_label": pred_lbl,
                "is_error": 1 if true_lbl != pred_lbl else 0,
                "split": split_name
            })

df = pd.DataFrame(all_results)
df = df.sort_values(by="image_id").reset_index(drop=True)

expected_len = len(new_hf_dataset["val"]) + len(new_hf_dataset["test"])
assert len(df) == expected_len, f"Row count mismatch! Expected {expected_len}, got {len(df)}"

save_path = "/content/Multimodal-AI-project/predictions.csv"
df.to_csv(save_path, index=False)
print(f"Success! predictions.csv saved to {save_path}. Row order locked.")

Starting inference...
Processing val split...
Processing test split...
Success! predictions.csv saved to /content/Multimodal-AI-project/predictions.csv. Row order locked.


In [ ]:
!git add train_model.py
!git commit -m "Added ResNet-50 training script"
!git push origin base_model

[base_model da49f26] Added ResNet-50 training script
 1 file changed, 67 insertions(+)
 create mode 100644 train_model.py
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
import pandas as pd

# 1. Load your generated predictions file
df = pd.read_csv("/content/Multimodal-AI-project/predictions.csv")

# 2. Separate validation and test splits
for split_name in ["val", "test"]:
    split_df = df[df["split"] == split_name]

    if len(split_df) > 0:
        # Since is_error is 1 for wrong and 0 for right,
        # the accuracy is simply 1 minus the average error rate rate.
        accuracy = (1 - split_df["is_error"].mean()) * 100

        print(f"--- {split_name.upper()} SPLIT METRICS ---")
        print(f"Total Samples: {len(split_df)}")
        print(f"Total Errors:  {split_df['is_error'].sum()}")
        print(f"Average Accuracy: {accuracy:.2f}%\n")

--- VAL SPLIT METRICS ---
Total Samples: 4715
Total Errors:  348
Average Accuracy: 92.62%

--- TEST SPLIT METRICS ---
Total Samples: 2358
Total Errors:  173
Average Accuracy: 92.66%

